## Registration — Ximea <-> Prime95B coordinate transform

Fits a coordinate transform mapping Ximea molecule positions onto the
Prime95B frame, using the linked single-molecule tables from
`Ximea_PostAnalysis.ipynb` / `Prime95B_PostAnalysis.ipynb`.

**FOV pairing assumption**: both cameras' acquisitions share the same
`Pos-<row>-<col>` stage-position grid (`power.txt`/the shared acquisition
session strongly suggest the stage was not moved between camera swaps), so a
Ximea FOV and a Prime95B FOV with the same `Pos-i-j` label are assumed to be
the same physical stage location, and each FOV's local `(xc, yc)` origin is
assumed to be the same relative point on the sample across positions (so
matched pairs from every FOV can be pooled into one global transform fit).
**Check the per-FOV match-count diagnostic below against real data** — if
match quality varies a lot between positions, this assumption may not hold
everywhere and per-FOV transforms may be needed instead.

Registration is done in **physical nm**, not raw pixel coordinates, so the
fitted transform captures real rotation/shear/translation/residual scale
mismatch rather than conflating it with the trivial 69 nm vs 110 nm
pixel-size ratio. Core matching/fitting logic: `src/RegistrationFunctions.py`
(ported from `pyFRETMiSeq/src/PreProcessingFunctions.py`'s
`_match_spot_pairs` + RANSAC affine fit).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
from pathlib import Path

import sys
sys.path.append('../..')

from src import IOFunctions
from src.RegistrationFunctions import match_spot_pairs, fit_affine_transform, save_transform

IO = IOFunctions.IO_Functions()


In [ ]:
# ── Paths and parameters ────────────────────────────────────────────────────────
XIMEA_FOLDER    = Path('/scratch/sycamore_asap_server/ASAP_Members_Other_Imaging_Data/Brendan/20260624_Ximea_vs_Prime95_beads/100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Ximea')
PRIME95B_FOLDER = Path('/scratch/sycamore_asap_server/ASAP_Members_Other_Imaging_Data/Brendan/20260624_Ximea_vs_Prime95_beads/100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Prime95B')

PIXEL_SIZE_XIMEA_NM    = 69.0
PIXEL_SIZE_PRIME95B_NM = 110.0

TRANSFORM_PATH = XIMEA_FOLDER.parent / 'ximea_to_prime95b_transform.csv'

# First-pass mutual-NN matching within each FOV (generous — the two FOVs
# aren't pre-aligned yet at this point) and RANSAC inlier threshold for the
# final affine fit, both in nm.
MAX_MATCH_DISTANCE_NM = 2000.0
RANSAC_RESIDUAL_NM    = 100.0

POS_RE = re.compile(r'Pos-\d+-\d+')


def fov_positions(folder, suffix):
    """{Pos-i-j label: path} for every '*<suffix>' file in folder."""
    out = {}
    for p in sorted(folder.glob(f'*{suffix}')):
        m = POS_RE.search(p.name)
        if m:
            out[m.group()] = p
    return out


ximea_pos    = fov_positions(XIMEA_FOLDER, '_linked_sm.h5')
prime95b_pos = fov_positions(PRIME95B_FOLDER, '_linked_sm.h5')
common_pos   = sorted(set(ximea_pos) & set(prime95b_pos))

print(f'[Ximea]    {len(ximea_pos)} linked FOVs')
print(f'[Prime95B] {len(prime95b_pos)} linked FOVs')
print(f'[Common]   {len(common_pos)} shared Pos-i-j labels')


In [ ]:
# ── Match single molecules within each shared FOV, in physical nm ──────────────
all_matched_src, all_matched_dst = [], []

for pos in common_pos:
    ximea_df    = IO.read_h5_database(str(ximea_pos[pos]))
    prime95b_df = IO.read_h5_database(str(prime95b_pos[pos]))

    ximea_nm    = ximea_df[['xc', 'yc']].to_numpy() * PIXEL_SIZE_XIMEA_NM
    prime95b_nm = prime95b_df[['xc', 'yc']].to_numpy() * PIXEL_SIZE_PRIME95B_NM

    matched_src, matched_dst = match_spot_pairs(ximea_nm, prime95b_nm, max_distance=MAX_MATCH_DISTANCE_NM)
    print(f'{pos}: Ximea={len(ximea_df):3d} molecules, Prime95B={len(prime95b_df):3d} molecules, '
          f'matched={len(matched_src):3d}')

    if len(matched_src):
        all_matched_src.append(matched_src)
        all_matched_dst.append(matched_dst)

all_matched_src = np.vstack(all_matched_src) if all_matched_src else np.zeros((0, 2))
all_matched_dst = np.vstack(all_matched_dst) if all_matched_dst else np.zeros((0, 2))
print(f'\nTotal pooled matched pairs: {len(all_matched_src)}')


In [ ]:
# ── Fit + save the registration transform ───────────────────────────────────────
tform, inliers = fit_affine_transform(
    all_matched_src, all_matched_dst, residual_threshold=RANSAC_RESIDUAL_NM,
)
print(f'Inliers: {int(inliers.sum())} / {len(all_matched_src)}')
print(f'Scale: {tform.scale}')
print(f'Rotation: {np.degrees(tform.rotation):.3f} deg')
print(f'Translation (nm): {tform.translation}')

pred = tform(all_matched_src)
resid = np.linalg.norm(pred - all_matched_dst, axis=1)
print(f'Residual (nm): mean={resid.mean():.1f}  median={np.median(resid):.1f}  max={resid.max():.1f}')

save_transform(tform, str(TRANSFORM_PATH))
print(f'Saved transform to {TRANSFORM_PATH}')


In [ ]:
# ── Diagnostic overlay: registered Ximea points vs Prime95B points ─────────────
registered_src = tform(all_matched_src)

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))

axs[0].scatter(all_matched_dst[:, 0], all_matched_dst[:, 1], s=10, alpha=0.6, label='Prime95B')
axs[0].scatter(registered_src[:, 0], registered_src[:, 1], s=10, marker='x', alpha=0.6, label='Ximea (registered)')
axs[0].set_xlabel('x / nm'); axs[0].set_ylabel('y / nm'); axs[0].legend()
axs[0].set_title('Registered overlay (all pooled pairs)')
axs[0].set_aspect('equal')

axs[1].hist(resid, 50)
axs[1].axvline(RANSAC_RESIDUAL_NM, color='k', ls='--', label='RANSAC threshold')
axs[1].set_xlabel('Residual / nm'); axs[1].set_ylabel('Count'); axs[1].legend()
axs[1].set_title('Registration residuals')

plt.tight_layout()
plt.show()
